In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/deepthism/ocean-waste-data/ocean-waste-data/trashcan/trashcan.yaml
/kaggle/input/datasets/deepthism/ocean-waste-data/ocean-waste-data/trashcan/labels/val/vid_000077_frame0000053.txt
/kaggle/input/datasets/deepthism/ocean-waste-data/ocean-waste-data/trashcan/labels/val/vid_000344_frame0000034.txt
/kaggle/input/datasets/deepthism/ocean-waste-data/ocean-waste-data/trashcan/labels/val/vid_000288_frame0000060.txt
/kaggle/input/datasets/deepthism/ocean-waste-data/ocean-waste-data/trashcan/labels/val/vid_000144_frame0000022.txt
/kaggle/input/datasets/deepthism/ocean-waste-data/ocean-waste-data/trashcan/labels/val/vid_000081_frame0000031.txt
/kaggle/input/datasets/deepthism/ocean-waste-data/ocean-waste-data/trashcan/labels/val/vid_000041_frame0000008.txt
/kaggle/input/datasets/deepthism/ocean-waste-data/ocean-waste-data/trashcan/labels/val/vid_000291_frame0000038.txt
/kaggle/input/datasets/deepthism/ocean-waste-data/ocean-waste-data/trashcan/labels/val/vid_000284_frame00

In [2]:
!pip install -q -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 4.4 MB/s eta 0:00:00


In [4]:
MODEL_NAME = "yolo26l"      
WEIGHTS    = "yolo26l.pt"   
IS_RTDETR  = False          # True only in the RT-DETR notebook

EPOCHS   = 100
PATIENCE = 30
IMGSZ    = 640
BATCH    = 16               
SEED     = 0

In [7]:
from pathlib import Path

NAMES = ['trash_bag', 'trash_bottle', 'trash_branch', 'trash_can', 'trash_clothing',
         'trash_container', 'trash_cup', 'trash_net', 'trash_pipe', 'trash_rope',
         'trash_snack_wrapper', 'trash_tarp', 'trash_unknown_instance', 'trash_wreckage']

def find_dir(name):
    for p in Path("/kaggle/input/datasets/deepthism/ocean-waste-data/ocean-waste-data").rglob(name):
        if p.is_dir() and (p / "images").exists():
            return p
    raise FileNotFoundError(name)

def write_yaml(path, root, train, val, test):
    with open(path, "w") as f:
        f.write(f"path: {root}\ntrain: {train}\nval: {val}\ntest: {test}\nnames:\n")
        for i, n in enumerate(NAMES):
            f.write(f"  {i}: {n}\n")

Y = Path("/kaggle/working/yaml"); Y.mkdir(exist_ok=True)
yaml_main = Y / "main.yaml"
yaml_dim  = Y / "dim.yaml"
yaml_dark = Y / "dark.yaml"

write_yaml(yaml_main, find_dir("trashcan"), "images/train", "images/val", "images/test")
write_yaml(yaml_dim,  find_dir("trashcan_dim"),  "images/test", "images/test", "images/test")
write_yaml(yaml_dark, find_dir("trashcan_dark"), "images/test", "images/test", "images/test")
print(open(yaml_main).read())

path: /kaggle/input/datasets/deepthism/ocean-waste-data/ocean-waste-data/trashcan
train: images/train
val: images/val
test: images/test
names:
  0: trash_bag
  1: trash_bottle
  2: trash_branch
  3: trash_can
  4: trash_clothing
  5: trash_container
  6: trash_cup
  7: trash_net
  8: trash_pipe
  9: trash_rope
  10: trash_snack_wrapper
  11: trash_tarp
  12: trash_unknown_instance
  13: trash_wreckage



In [ ]:
import os
os.environ["WANDB_MODE"] = "disabled"
from ultralytics import YOLO, RTDETR

model = RTDETR(WEIGHTS) if IS_RTDETR else YOLO(WEIGHTS)
model.train(
    data=str(yaml_main), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ,
    batch=BATCH, seed=SEED, workers=4, device=0,
    project="/kaggle/working/runs", name=MODEL_NAME, exist_ok=True, plots=True,
)
save_dir = Path(model.trainer.save_dir)
print("Saved to:", save_dir)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.171 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/yaml/main.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


      2/100      11.1G      1.398      2.818    0.01252         23        640: 100% ━━━━━━━━━━━━ 222/222 1.6it/s 2:200.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 24/24 2.5it/s 9.6s0.5s
                   all        759        885      0.666      0.306      0.373       0.23

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      3/100        11G      1.481      2.335    0.01373         28        640: 100% ━━━━━━━━━━━━ 222/222 1.6it/s 2:180.5ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 24/24 2.5it/s 9.6s0.5s
                   all        759        885       0.45      0.452      0.447      0.279

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      4/100      11.1G      1.477      2.052    0.01361         29        640: 100% ━━━━━━━━━━━━ 222/222 1.6it/s 2:170.4ss
                 Class     Images